In [ ]:
import os
import numpy as np 
import pandas as pd 
import transition as tr
import segmentation as sg
import adjective_mean as am
import validation_model as vm
import dbscan_clustering as db
import adjectives_handle as ah
import data_processing.src.plot as plot

from sklearn.decomposition import PCA
from data_processing.src import config

# 0. Initial Parameters Selection 

1. Choose the Target Activity to be learned: "sphereActivity", "ladderActivity", "trashActivity", "pilotActivity".
2. Select the Segmentation Threshold: it is a *trail and error* approach based on the expected number of segments. 
3. Select the Normalization Scaler.
4. Choose the Features on which to perform training. 

In [ ]:
target_activity = "sphereActivity"
threshold = 0.6
scaler = "standard"

# only rotation
"""result_directory = "dbscan_results_rotation"
features = config.ROTATION_FEATURES
segmentation_result = config.ROTATION_SEGMENTATION_DIR 
processed_data_path = config.PROCESSED_DATA_DIR
histogram_dir = config.ROTATION_HISTOGRAMS_DIR"""

# only position
"""result_directory = "dbscan_results_position"
features = config.POSITION_FEATURES
segmentation_result = config.POSITION_SEGMENTATION_DIR 
processed_data_path = config.PROCESSED_DATA_DIR
histogram_dir = config.POSITION_HISTOGRAMS_DIR"""

# both rotation and position
result_directory = "dbscan_results"
features = config.FEATURES
segmentation_result = config.BOTH_SEGMENTATION_DIR 
processed_data_path = config.PROCESSED_DATA_DIR
histogram_dir = config.BOTH_HISTOGRAMS_DIR

feature_len = len(features) - 2 

# Model Training

The training is divided into three steps: 
1. Segmentation of the Target Action into Elementary Actions.
2. Clustering of Elementary Actions.
3. Processing of the Markov Chain.

## 1 Target Activity Segmentation

1. Perform the Segmentation process and evaluate the histogram.
2. Adjust the *Threshold* (Step 0) to obtain an appropriate number of segments per log.

In [ ]:
rsv_df, lsv_all_user = sg.segment_all_users(
            activity=target_activity, 
            features=features,
            threshold=threshold, 
            scaler=scaler, 
            filepath=segmentation_result
        )

os.makedirs(segmentation_result + f"/segments_number/{target_activity}", exist_ok=True)
os.makedirs(histogram_dir + f"/{target_activity}", exist_ok=True)

plot.plot_histogram_of_segments(
    filepath=segmentation_result + f"/segments_number/{target_activity}/{scaler}/segmentation_results_{target_activity}_{threshold}_{scaler}.csv",
    savepath=histogram_dir + f"/{target_activity}")


## 2 Elementary Actions Clustering 

Apply the DBSCAN clustering algorithm to define the most generalised representation of each elementary action (*substantives*).

In [ ]:
rsv_path = segmentation_result + f"/right_singular_vector/{target_activity}/{scaler}/rsv_{target_activity}_{threshold}_{scaler}.csv"

In [ ]:
print(f"===== Processing activity: {target_activity}, threshold: {threshold}, scaler: {scaler} =====\n")

df_origin = pd.read_csv(rsv_path, index_col=None)

df = df_origin.drop(columns=['Username', "Adjective", 'LogNumber'])
df = df.dropna()
df = df.drop_duplicates()

print("=================================================\n")

### 2.1 Initial Plots 

In [ ]:
#db.initial_plot(df)

### 2.2 GRID for Hyperparameters

Choose the best combination of the hyperparameters following these rules: 
1. Avoid the presence of a cluster that is considerably larger than the others.
2. Avoid having more than half of all elementary actions being outliers.
3. Accept solutions in which there are at least two clusters that are larger than the others.
4. Choose solutions with an appropriate number of clusters (neither too few nor too many).

If you are unsure between two solutions, choose the one that produces fewer clusters.

In [ ]:
pca = PCA(n_components=0.9)

X_pca = pca.fit_transform(df.values)

# Trasforma in DataFrame con indici originali
df_pca = pd.DataFrame(X_pca, index=df.index, columns=[f'{i+1}' for i in range(X_pca.shape[1])])

print("== DBSCAN Clustering Results ==\n")
n_cols = len(df_pca.columns)
length = len(df_pca)
print(f"Number of columns: {n_cols}\nLength of DataFrame: {length}\n")
db.grid(df_pca)

### 2.3 Hyperparameter Selection and Clustering Execution 

Enter the chosen values for *eps* and *min_samples* (Step 2.2).

In [ ]:
eps = 0.25
min_samples = 8

In [ ]:
best_labels = db.dbscan_clustering(df_pca, eps=eps, min_samples=min_samples)
df['Label'] = best_labels
df_pca['Label'] = best_labels

#db.pair_plot_cluster(df_pca)

# add the labels to df_origin handling the duplicates rows 
feature_cols = df_origin.columns[-18:].to_list()

df_origin = df_origin.merge(
    df,
    on=feature_cols,
    how="left"
)

save_dir = processed_data_path + f"/{result_directory}/{target_activity}/{scaler}/all_points/dbscan_{target_activity}_{threshold}_{scaler}_eps{eps}_minsample{min_samples}.csv"

os.makedirs(os.path.dirname(save_dir), exist_ok=True)
df_origin.to_csv(save_dir, index=False)

df_no_outliers = df.drop(df[df['Label'] == -1].index)
unique_labels = df_no_outliers['Label'].unique()

print(f"Number of unique labels (excluding outliers): {len(unique_labels)}\n")

### 2.4 Printing and Saving Results.

The set of elementary actions is the *dictionary of substantives*.

Each *substantives* is characterised by an *adjective* that represents its speed. Together, they make up the *dictionary of adjectives*.

In [ ]:
df_substantives = db.get_substantives_dataframe(df=df_no_outliers, unique_labels=unique_labels)
print("\n== Substantives DataFrame ==\n")
display(df_substantives)

save_dir = processed_data_path + f"/{result_directory}/{target_activity}/{scaler}/substantives/substantives_{target_activity}_{threshold}_{scaler}_eps{eps}_minsample{min_samples}.csv"
os.makedirs(os.path.dirname(save_dir), exist_ok=True)

df_substantives.to_csv(processed_data_path + f"/{result_directory}/{target_activity}/{scaler}/substantives/substantives_{target_activity}_{threshold}_{scaler}_eps{eps}_minsample{min_samples}.csv", index=False)

## 2.5 Adjectives Dictionary Management

In the segmentation in Section 1, all left singular vectors (lsv) corresponding to right singular vectors (rsv) were also retrieved. 
*lsv_all_user* is a multiple dictionary in which the first key is the username, the second key is the log number, and the third key discriminates between the various lsv. 
Now we need to manipulate this information:
- **ah.adjective_dict()**. This allows us to create a dictionary that groups the various lsv according to the cluster to which the associated right singular vector belongs. 
- **ah.adjectives_segmentation()**. This allows you to segment the lsv in each cluster into four possible categories (slow_pos, slow_neg, fast_pos, fast_neg) based on two thresholds set at 0.2 and -0.2. The resulting dict will have the cluster as the first key and the category as the second key. Each category contains all the lsv belonging to it.
- **ah.generalize_adjectives()**. Generalises the left singular vectors of each cluster and each category with a value. At the end, we obtain a DataFrame where each row contains the cluster, the category and the representative value: each cluster will have at most 4 adjectives, one per category.


In [ ]:
# Create LSV clusters dictionary based on DBSCAN results
df_new = df_origin.drop(columns=['Adjective']).drop_duplicates()

lsv_clusters_dict = ah.adjectives_dict(best_labels, lsv_all_user, df_new)

clusters_dict = ah.adjectives_segmentation(lsv_clusters_dict)

df_adjectives = ah.generalize_adjectives(clusters=clusters_dict)

print("\n== Adjectives DataFrame ==\n")
display(df_adjectives)

save_dir = processed_data_path + f"/{result_directory}/{target_activity}/{scaler}/adjectives/"
os.makedirs(os.path.dirname(save_dir), exist_ok=True)

df_adjectives.to_csv(save_dir + f"adjectives_{target_activity}_{threshold}_{scaler}_eps{eps}_minsample{min_samples}.csv", index=False)

## 3. Markov Chain: States Transactions

A state in the *Markov Chain* is given by the pair of a *substantive* and one of its corresponding *adjectives*.

Define all possible states and calculate the probability of each state transition based on how the elementary actions obtained from the training data follow one another.

In [ ]:
_, _, df_initial = tr.transition_computation(
        activity=target_activity,
        scaler=scaler,
        threshold=threshold,
        eps=eps,
        min_samples=min_samples,
        dbscan_results=result_directory,
        file_path=processed_data_path + f"/{result_directory}/{target_activity}/{scaler}"
    )

display(df_initial)

# Model Validation 

1. Select the user from whom you want to retrieve the activity on which to validate the model.
2. Select the activity on which to validate.

In [ ]:
username = "grims3"
test_activity = "sphereActivity"

#file_path = config.find_file_from_username(username)
file_path = config.find_file_from_username_and_basepath(username=username, base_path=config.RAW_DATA_DIR + "/test_data")
print(file_path)

## 1. Test Activity Segmentation

Select the threshold for segmentation.

Note: if the test activity is the same as the train activity, choose the same threshold as the train (Step 0).

In [ ]:
new_threshold = 0.6

print(f"===== Segmentating user: {username}, test_activity: {test_activity}, scaler: {scaler}, threshold: {new_threshold} =====")
rsv_df, lsv_user = vm.segment_user(file_path=file_path, features=features, activity=test_activity, scaler=scaler, threshold=new_threshold)

print("\n== RSV DataFrame: ==")
display(rsv_df)


In [ ]:
for x in range(1, len(rsv_df['LogNumber'].unique()) + 1):
    print(f"LogNumber: {x}, Number of segments: {len(rsv_df[rsv_df['LogNumber'] == x])}")

## 2. Maximum Likelihood Classifier

### 2.1 Classify the new Elementary Actions

In [ ]:
file_path = processed_data_path + f"/{result_directory}/{target_activity}/{scaler}" 

# Calcola le statistiche per ogni classe
df = pd.read_csv(file_path + f"/all_points/dbscan_{target_activity}_{threshold}_{scaler}_eps{eps}_minsample{min_samples}.csv")
df = df.drop(columns=['Adjective']).drop_duplicates()

feature_columns = df.columns[-feature_len - 1: -1]  

df_inliers = df[df["Label"] != -1]

class_stats = vm.compute_class_stats(df=df_inliers, feature_columns=feature_columns, label_column="Label", epsilon=1e-6)
class_threshold = vm.compute_threshold(df=df_inliers, feature_columns=feature_columns, class_stats=class_stats, quantile=0.05)

print("class_threshold: ", class_threshold)

i = 0
print("\n== Classify New Points ==")
for index, row in rsv_df.iterrows():
    point = row[-feature_len:].to_numpy()

    label = vm.classify_point(point, class_stats, threshold=class_threshold)

    if index != 0 and row['LogNumber'] != prev_label:
        i = 1
    else: 
        i += 1

    prev_label = row['LogNumber']
    #print(f"LogNumber: {row['LogNumber']} - Elementary Action: {i} - Classified as: {label}\n")

log_df = pd.DataFrame({
    'Username': username,
    'LogNumber': rsv_df['LogNumber'],
    'Label': [vm.classify_point(row[-feature_len:].to_numpy(), class_stats, threshold=class_threshold) for _, row in rsv_df.iterrows()]
})

print("log_df: \n", log_df)



### 2.2 Identify the Adjectives that Correspond to the Substantives -> States

In [ ]:
adj_df = pd.read_csv(file_path + f"/adjectives/adjectives_{target_activity}_{threshold}_{scaler}_eps{eps}_minsample{min_samples}.csv")

# Wrap lsv_user with username as the first key
lsv_user_with_username = {username: lsv_user} 

lsv_mean = ah.calculate_segment_means(lsv_user_with_username)

df_mean = vm.compute_user_state(lsv_mean, log_df, adj_df)

display(df_mean)

## 3 Computation of the Average Log-Likelihood for Each Transition (Average Self-Information)

Calculation of the probability that the sequence of elementary actions performed by the test user based on the state transition matrix calculated in Step 3.

---- 

There are sequences of different lengths for both the same Complex Action and for different Complex Actions. 

Simply calculating the probability of each sequence creates a problem: longer sequences will independently have a lower probability because we are multiplying probabilities (0 < p < 1). 

To solve this problem, we calculate the **logarithmic average of the probabilities** . This is a measure of the quality of a sequence normalised for its length. 

----

When calculating the probability of each sequence, we take into account the number of **Ghost States** relative to the total states and penalise the probability based on these values using the **Ghost Function**:

    G(n, N) = exp(-(n^4)/N)

-n: number of Ghost States

-N: total number of states in the sequence


In [ ]:
ghost_exp = 4

prob_matrix_df = pd.read_csv(config.RESULTS_DIR + f"/transition_results/{target_activity}/{scaler}/transition_probabilities_{target_activity}_{threshold}_{scaler}_eps{eps}_minsample{min_samples}.csv", index_col=0)
initial_probs_df = pd.read_csv(config.RESULTS_DIR + f"/transition_results/{target_activity}/{scaler}/initial_probabilities_{target_activity}_{threshold}_{scaler}_eps{eps}_minsample{min_samples}.csv")   

ghost_end_results = []
for log_number, group in df_mean.groupby("LogNumber"):
    print(f"\n== GHOST STATE: Processing LogNumber: {log_number} ==")
    prob, N_states, ghost_count = vm.sequence_probability_ghost_state_end(group_df=group, prob_matrix_df=prob_matrix_df, initial_probs_df=initial_probs_df, exp=ghost_exp)
    ghost_end_results.append((test_activity, log_number, N_states, ghost_count, prob))

df_sequence_ghost_end = pd.DataFrame(ghost_end_results, columns=["TestActivity", "LogNumber", "N_states", "Ghost_Count", "Sequence_Probability"], index=None)

df_sequence_ghost_end["Normalized"] = (df_sequence_ghost_end["Sequence_Probability"] / df_sequence_ghost_end["Sequence_Probability"].sum())   

# valori molto negativi sono transizioni molto improbabili 
df_sequence_ghost_end["LogProb"] = np.log(df_sequence_ghost_end["Sequence_Probability"].replace(0, np.nan))
df_sequence_ghost_end["Length"] = df_mean.groupby("LogNumber").size().values
df_sequence_ghost_end["AvgLogProb"] = df_sequence_ghost_end["LogProb"] / df_sequence_ghost_end["Length"]


# Printing and Saving Results 

The values are negative: 
- The more negative AvgLogProb is, the less likely the transition is. 
- The closer AvgLogProb is to 0, the more likely the transition is.

--- 

NOTE: The presence of _"AvgLogProb = -inf"_ indicates the presence of a transition that never occurred during model training. The more data there is to train the model, the better the coverage of actual state transitions and the greater the detail of the Markov chain.

In [ ]:
columns = ["Username", "Features", "Target_Activity", "TestActivity", "Target_Threshold", "Test_Threshold", "eps", "min_samples", "scaler", "LogNumber", "N_states", "Ghost_Count", "Ghost_Exp", "Sequence_Probability", "Normalized", "LogProb", "Length", "AvgLogProb"]

if features == config.ROTATION_FEATURES: 
    feat = "rotation"
elif features == config.POSITION_FEATURES:
    feat = "position"
else:
    feat = "rotation_position"
# Create list to store all rows
rows_to_add = []

for index, row in df_sequence_ghost_end.iterrows():
    rows_to_add.append({
        "Username": username,
        "Features": feat,
        "Target_Activity": target_activity,
        "TestActivity": test_activity,
        "Target_Threshold": threshold,
        "Test_Threshold": new_threshold,
        "eps": eps,
        "min_samples": min_samples,
        "scaler": scaler,
        "LogNumber": int(row["LogNumber"]),
        "N_states": row["N_states"],
        "Ghost_Count": row["Ghost_Count"],
        "Ghost_Exp": ghost_exp,
        "Sequence_Probability": row["Sequence_Probability"],
        "Normalized": (row["Normalized"] if not pd.isna(row["Normalized"]) else 0.0),
        "LogProb": (row["LogProb"] if not pd.isna(row["LogProb"]) else -np.inf),
        "Length": row["Length"],
        "AvgLogProb": (row["AvgLogProb"] if not pd.isna(row["AvgLogProb"]) else -np.inf)
    })

df_save = pd.DataFrame(rows_to_add, columns=columns)

path = config.RESULTS_DIR + f"/recognition_results/results_lsv.csv"
os.makedirs(os.path.dirname(path), exist_ok=True)

file_exists = os.path.exists(path)
write_header = not file_exists
df_save.to_csv(path, header=write_header, mode='a', index=False)

display(df_save)

# Check for and remove duplicates from the CSV file
df_loaded = pd.read_csv(path)

# check for duplicates
duplicates_count = df_loaded.duplicated().sum()

if duplicates_count > 0:
    # remove duplicates and save back 
    df_cleaned = df_loaded.drop_duplicates()
    df_cleaned.to_csv(path, index=False)